# 02. Baseline и первые модели

Здесь обучаются простые модели для CP1. Test используется только один раз после выбора лучшей модели по validation RMSE.

In [19]:
from pathlib import Path
import os

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TARGET = "traffic_volume"

NUMERIC_FEATURES = [
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "hour",
    "day_of_week",
    "month",
    "year",
    "is_weekend",
    "is_rush_hour",
]

CATEGORICAL_FEATURES = ["holiday", "weather_main", "weather_description"]

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "Metro_Interstate_Traffic_Volume.csv"


def load_traffic_data(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)


def clean_traffic_data(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data["date_time"] = pd.to_datetime(data["date_time"], errors="coerce")
    data = data.dropna(subset=["date_time", TARGET])
    data["holiday"] = data["holiday"].fillna("None")
    data = data.drop_duplicates()
    data.loc[data["temp"] <= 1, "temp"] = np.nan
    data = data.sort_values("date_time").reset_index(drop=True)
    return data


def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data["hour"] = data["date_time"].dt.hour
    data["day_of_week"] = data["date_time"].dt.dayofweek
    data["month"] = data["date_time"].dt.month
    data["year"] = data["date_time"].dt.year
    data["is_weekend"] = data["day_of_week"].isin([5, 6]).astype(int)
    data["is_rush_hour"] = data["hour"].isin([7, 8, 9, 16, 17, 18]).astype(int)
    return data


def time_based_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    data = df.sort_values("date_time").reset_index(drop=True)
    train = data[data["date_time"] < "2017-01-01"].copy()
    validation = data[(data["date_time"] >= "2017-01-01") & (data["date_time"] < "2018-01-01")].copy()
    test = data[data["date_time"] >= "2018-01-01"].copy()
    return train, validation, test


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor() -> ColumnTransformer:
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, NUMERIC_FEATURES),
            ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
        ]
    )


def build_model_pipeline(estimator) -> Pipeline:
    return Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            ("model", estimator),
        ]
    )


def get_model_candidates() -> dict[str, Pipeline]:
    return {
        "Dummy mean": build_model_pipeline(DummyRegressor(strategy="mean")),
        "Ridge Regression": build_model_pipeline(Ridge()),
        "Random Forest": build_model_pipeline(
            RandomForestRegressor(
                n_estimators=80,
                max_depth=20,
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=1,
            )
        ),
        "HistGradientBoosting": build_model_pipeline(
            HistGradientBoostingRegressor(
                max_iter=200,
                learning_rate=0.08,
                random_state=RANDOM_STATE,
            )
        ),
    }


def regression_metrics(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }


def evaluate_model(model: Pipeline, x: pd.DataFrame, y: pd.Series) -> dict[str, float]:
    predictions = model.predict(x)
    return regression_metrics(y, predictions)

## Данные и временной split

Random split не используется. Для временного ряда важно обучаться на прошлом и проверяться на будущем, иначе появится data leakage.

- train: `date_time < 2017-01-01`
- validation: `2017-01-01 <= date_time < 2018-01-01`
- test: `date_time >= 2018-01-01`

In [20]:
raw_df = load_traffic_data(DATA_PATH)
df = add_time_features(clean_traffic_data(raw_df))

train_df, valid_df, test_df = time_based_split(df)

split_summary = pd.DataFrame(
    {
        "part": ["train", "validation", "test"],
        "rows": [len(train_df), len(valid_df), len(test_df)],
        "start": [train_df["date_time"].min(), valid_df["date_time"].min(), test_df["date_time"].min()],
        "end": [train_df["date_time"].max(), valid_df["date_time"].max(), test_df["date_time"].max()],
    }
)

display(split_summary)
print(f"RANDOM_STATE = {RANDOM_STATE}")

,part,rows,start,end
0,train,29643,2012-10-02 09:00:00,2016-12-31 23:00:00
1,validation,10596,2017-01-01 00:00:00,2017-12-31 23:00:00
2,test,7948,2018-01-01 00:00:00,2018-09-30 23:00:00


RANDOM_STATE = 42


In [21]:
feature_columns = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X_train = train_df[feature_columns]
y_train = train_df[TARGET]
X_valid = valid_df[feature_columns]
y_valid = valid_df[TARGET]
X_test = test_df[feature_columns]
y_test = test_df[TARGET]

print(f"Числовые признаки: {NUMERIC_FEATURES}")
print(f"Категориальные признаки: {CATEGORICAL_FEATURES}")
print(f"Всего признаков для модели: {len(feature_columns)}")

assert TARGET not in feature_columns
assert "date_time" not in feature_columns

Числовые признаки: ['temp', 'rain_1h', 'snow_1h', 'clouds_all', 'hour', 'day_of_week', 'month', 'year', 'is_weekend', 'is_rush_hour']
Категориальные признаки: ['holiday', 'weather_main', 'weather_description']
Всего признаков для модели: 13


## Метрики

Основная метрика - RMSE. Она сильнее штрафует большие ошибки, поэтому хорошо подходит, если крупные промахи нежелательны. Дополнительно считаются MAE и R2. MAE проще интерпретировать как среднюю абсолютную ошибку прогноза в машинах в час.

## Обучение моделей

Для всех моделей используется `Pipeline` и `ColumnTransformer`. Числовые признаки проходят `SimpleImputer(strategy="median")` и `StandardScaler`, категориальные - `SimpleImputer(strategy="most_frequent")` и `OneHotEncoder(handle_unknown="ignore")`.

In [22]:
models = get_model_candidates()
validation_results = []
fitted_models = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    metrics = evaluate_model(model, X_valid, y_valid)
    validation_results.append({"model": model_name, **metrics})
    fitted_models[model_name] = model

results_df = pd.DataFrame(validation_results).sort_values("RMSE").reset_index(drop=True)
display(results_df[["model", "RMSE", "MAE", "R2"]].round({"RMSE": 2, "MAE": 2, "R2": 4}))

,model,RMSE,MAE,R2
0,HistGradientBoosting,491.71,315.11,0.9387
1,Random Forest,495.90,314.14,0.9377
2,Ridge Regression,1574.71,1347.07,0.3717
3,Dummy mean,1989.58,1748.43,-0.0030


In [23]:
best_model_name = results_df.loc[0, "model"]
print(f"Лучшая модель по validation RMSE: {best_model_name}")

Лучшая модель по validation RMSE: HistGradientBoosting


По validation выбираем модель с минимальным RMSE. Лучший результат у `HistGradientBoosting`: RMSE 491.71, MAE 315.11, R2 0.9387. Random Forest получился очень близко и немного лучше по MAE, но выбор делается по RMSE. Test не использовался для выбора модели или настройки параметров.

In [24]:
X_train_valid = pd.concat([X_train, X_valid], axis=0)
y_train_valid = pd.concat([y_train, y_valid], axis=0)

best_model = clone(models[best_model_name])
best_model.fit(X_train_valid, y_train_valid)

test_metrics = evaluate_model(best_model, X_test, y_test)
test_results_df = pd.DataFrame([{"model": best_model_name, **test_metrics}])
display(test_results_df[["model", "RMSE", "MAE", "R2"]].round({"RMSE": 2, "MAE": 2, "R2": 4}))

,model,RMSE,MAE,R2
0,HistGradientBoosting,472.28,270.89,0.9427


## Выводы

DummyRegressor нужен как наивная точка отсчета. Ridge Regression заметно лучше baseline, но деревья и бустинг лучше ловят нелинейность. На test выбранный `HistGradientBoosting` получил RMSE 472.28, MAE 270.89 и R2 0.9427. Финальная оценка сделана один раз на test после выбора модели по validation RMSE.